# Script using the nnUNet framework

Steps:
- Set up paths and environment variables

- Prepare dataset metadata (dataset.json)

- Run preprocessing

- Train the model

- Run inference

- Evaluate results

In [1]:
import os
import sys
import nnunetv2
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import json
import pandas as pd
from nnunetv2.dataset_conversion.generate_dataset_json import generate_dataset_json
from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import (
    extract_fingerprint_entry,
    plan_experiment_entry,
    preprocess_entry,
    plan_and_preprocess_entry
)

from nnunetv2.run.run_training import run_training_entry

from nnunetv2.inference.predict_from_raw_data import predict_entry_point

from nnunetv2.evaluation.evaluate_predictions import evaluate_folder_entry_point

from functions.custom_prediction_and_eval import custom_prediction_and_evaluation, display_prediction_graphs


### 1) Setup environment


In [2]:
# ---------------------------------------------------------
# 1) Set correct nnUNetv2 environment variables
# ---------------------------------------------------------

# os.environ["nnUNet_raw"] = "/data/leuven/368/vsc36890/User/nnUnet/nnUNet_raw"
# os.environ["nnUNet_preprocessed"] = "/data/leuven/368/vsc36890/User/nnUnet/nnUNet_preprocessed"
# os.environ["nnUNet_results"] = "/data/leuven/368/vsc36890/User/nnUNet/nnUNet_results"

os.environ["nnUNet_raw"] = "/Users/User/nnUNet/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/Users/User/nnUNet/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/Users/User/nnUNet/nnUNet_results"


print("nnUNet_raw =", os.environ.get("nnUNet_raw"))
print("nnUNet_preprocessed =", os.environ.get("nnUNet_preprocessed"))
print("nnUNet_results =", os.environ.get("nnUNet_results"))

nnUNet_raw = /Users/User/nnUNet/nnUNet_raw
nnUNet_preprocessed = /Users/User/nnUNet/nnUNet_preprocessed
nnUNet_results = /Users/User/nnUNet/nnUNet_results


In [3]:
# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
task_id = 195


config = "2d"                                   # "2d", "3d_fullres", etc.
folds = "0"                                     # e.g. "0", "1", "0,1,2,3,4", "all"
planner = "Fixed4StagePlanner"                  # nnUNetPlannerResEncL - nnUNetPlannerResEncM - Fixed4StagePlanner - Fixed3StagePlanner
planner_json_name = "Fixed4StagePlannerPlans"   # nnUNetResEncUNetSPlans - nnUNetResEncUNetMPlans - Fixed4StagePlannerPlans - Fixed3StagePlannerPlans
trainer = "nnUNetTrainer"

channel_name = f"{task_id:03d}_SDO_AIA"
task_name = f"Dataset{channel_name}"

json_output_folder = os.path.join(os.environ["nnUNet_raw"], task_name)

pof = "/Users/User/nnUNet/nnUNet_predictions"

### 2) Create dataset.json

After reading documentation, the name of the channel just determins the normalisation that is gonna be used. So, need to try "CT" and "193_SDO_AIA" to cross the different normalization. (see: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/explanation_normalization.md)

In [33]:
##This create the Jason file for nnUnet to understand what to do with the dataset
generate_dataset_json(
    output_folder=json_output_folder,
    channel_names={0: "193_SDO_AIA"},                   # Defining the names of the images. Can have multiple picture in channel: channel_names = {0: "T1", 1: "T2"}. the name will determine the normalization: try with "CT" and "193_SDO_AIA" 
    labels={"background": 0, "Coronal hole": 1},       # Defining what the values in the Mask (to train). Can multiclass: labels={"background": 0, "liver": 1, "tumor": 2, "vessel": 3}
    num_training_cases=7872,                            # Nbr of cases
    file_ending=".nii.gz",
    license='SDO Data: NASA/SDO, LMSAL. Please cite NASA/SDO in publications.',
    converted_by= "Romain MADDOX, Astrophysics Master Student at KULeuven"
)

print(f"dataset.json created in {json_output_folder}")

dataset.json created in /Users/User/nnUNet/nnUNet_raw\Dataset194_SDO_AIA


- Exemple os strucutre:

nnUNet_raw/  
└── Dataset005_MyDataset/  
......├── imagesTr/     # Training images  
............└── case_0000_0000.nii.gz   # channel 0 = T1  
............└── case_0000_0001.nii.gz   # channel 1 = T2  
............└── case_0001_0000.nii.gz   # channel 0 = T1  
............└── case_0001_0001.nii.gz   # channel 1 = T2  
  
......├── labelsTr/     # Training segmentation labels  
............└── case_0000.nii.gz  
............└── case_0001.nii.gz  
  
......├── imagesTs/     # Test images (no labels here)  
............└── case_0002_0000.nii.gz  
............└── case_0002_0001.nii.gz  
  
......└── dataset.json  # Metadata (created by generate_dataset_json)  



### 3) Plan and preprocess data (optional since triggered automatically in step 4)

In [ ]:
sys.argv = ["extract_fingerprint", "-d", str(task_id)]
extract_fingerprint_entry()

In [4]:
sys.argv = ["plan_and_preprocess", "-d", str(task_id), "-pl", planner]
plan_and_preprocess_entry()

# To run the steps individually instead of together:
#
# plan_experiment_entry(["plan_experiment", "-d", str(task_id), "-pl", planner])
# preprocess_entry(["preprocess", "-d", str(task_id)])

Fingerprint extraction...
Dataset194_SDO_AIA
Experiment planning...
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 120, 'patch_size': [256, 256], 'median_image_size_in_voxels': array([256., 256.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization'], 'use_mask_for_norm': [False], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape', 'resampling_fn_probabilities_kwargs': {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'architecture': {'network_class_name': 'dynamic_network_architectures.architectures.unet.PlainConvUNet', 'arch_kwargs': {'n_stages': 4, 'feat

100%|██████████| 7872/7872 [07:47<00:00, 16.84it/s]


Configuration: 3d_fullres...
INFO: Configuration 3d_fullres not found in plans file nnUNetPlans.json of dataset Dataset194_SDO_AIA. Skipping.
Configuration: 3d_lowres...
INFO: Configuration 3d_lowres not found in plans file nnUNetPlans.json of dataset Dataset194_SDO_AIA. Skipping.


### 4) Train the model

I used this code on the HPC. <br>
Param:<br>
Cluster: wice<br>
Toolchain and Python versions: 2023 and Python/3.11.3-GCCcore-12.3.0 (check -> Load the SciPy-bundle module with FOSS)<br>
Partition: gpu_a100<br>
Account: X<br>
Number of hours: X<br>
Number of nodes: 1<br>
Number of processes per node: 1
<br>Required memory per core in megabytes: 64000
<br>Number of GPUs: 1
<br>Pre-run scriptlet:
<br>module load SciPy-bundle/2023.07-gfbf-2023a libjpeg-turbo/2.1.5.1-GCCcore-12.3.0 libpng/1.6.39-GCCcore-12.3.0 imageio/2.33.1-gfbf-2023a matplotlib/3.7.2-gfbf-2023a tqdm/4.66.1-GCCcore-12.3.0 scikit-learn/1.3.1-gfbf-2023a CUDA/12.1.1 cuDNN/8.9.2.26-CUDA-12.1.1 PyTorch/2.1.2-foss-2023a-CUDA-12.1.1 PyTorch-bundle/2.1.2-foss-2023a-CUDA-12.1.1 torchvision/0.16.0-foss-2023a-CUDA-12.1.1 Triton/2.1.0-foss-2023a-CUDA-12.1.1

In [ ]:
#verify name of JSON file

# sys.argv = [
#     "run_training",          # script name (ignored)
#     str(task_id),        # dataset_name_or_id
#     config,                  # configuration
#     folds,                   # fold
#     "-tr", trainer,          # trainer class
#     "-p", planner_json_name,           # planner class
#     "-device", "cpu"         # use CPU
# ]


# # Call the entrypoint
# run_training_entry()

# # -----------------------------
# # Put this in front
# #
# # from torch.cuda.amp import GradScaler
# #
# # self.grad_scaler = GradScaler() if self.device.type == 'cuda' else None

### 5) Create predictions on the test images (imagesTs) or training images (imagesTr)

Set parameters:

In [ ]:
# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
task_id = 201

config = "2d"                                   # "2d", "3d_fullres", etc.
folds = "0"                                     # e.g. "0", "1", "0,1,2,3,4", "all"
planner = "Fixed3StagePlanner"                  # nnUNetPlannerResEncL - Fixed5StagePlanner - Fixed4StagePlanner - Fixed3StagePlanner
planner_json_name = "Fixed3StagePlannerPlans"   # nnUNetResEncUNetSPlans - Fixed5StagePlannerPlans - Fixed4StagePlannerPlans - Fixed3StagePlannerPlans
trainer = "nnUNetTrainer_1000epochs_crossval_with_ifb125_loss"                       # nnUNetTrainer_250epochs_tempval - nnUNetTrainer_200epochs_tempval - nnUNetTrainer_150epochs_tempval - nnUNetTrainer_100epochs_tempval

channel_name = f"{task_id:03d}_SDO_AIA"
task_name = f"Dataset{channel_name}"

json_output_folder = os.path.join(os.environ["nnUNet_raw"], task_name)
pof = "/Users/User/nnUNet/nnUNet_predictions"

channel_name = f"{task_id:03d}_SDO_AIA"
task_name = f"Dataset{channel_name}"
json_output_folder = os.path.join(os.environ["nnUNet_raw"], task_name)
pof = "/Users/User/nnUNet/nnUNet_predictions"

list_name_chk = ["checkpoint_best"]
tf_with_tta = True
type_mask = ["spoca"]   # region - spoca

In [ ]:
custom_prediction_and_evaluation(
    list_chk = list_name_chk,
    tf_with_tta = tf_with_tta,
    type_mask = type_mask,

    trainer = trainer,
    planner_json_name = planner_json_name,
    config = config,
    json_output_folder = json_output_folder,
    pof = pof,
    task_name = task_name,
    task_id = task_id,
    folds = folds
    is_imagesTs= True
)

Starting...

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

perform_everything_on_device=True is only supported for cuda devices! Setting this to False
There are 864 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 864 cases that I would like to predict
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
File =checkpoint_epoch_50 with tta ==True regarding the spoca masks Saved.

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., 

# 6) Evaluation of the predicted 

In [ ]:
from functions.metric_calculation_tools import calculate_metrics_per_pixel, calculate_metrics_per_structure, calcuate_APa50

In [ ]:
# paths to predictions and gt 
pred_folder = r"C:\Users\User\nnUNet\nnUNet_predictions\Dataset201_SDO_AIA\nnUNetTrainer_1000epochs_crossval_with_ifb125_loss__Fixed3StagePlannerPlans__2d\spoca\checkpoint_best_with_tta" #_of_imagesTr
gt_folder   = r"C:\Users\User\nnUNet\nnUNet_raw\Dataset364_SDO_AIA\labelsTs\spoca"

## for the Val, use the datasplit.json file to fide the validation class:
# with open(r"C:\Users\User\nnUNet\nnUNet_preprocessed\Dataset364_SDO_AIA\splits_final_TEMPORAL_364.json", "r") as f:
#     vec = json.load(f)


# The classes wanted (0 is the background)
classes = [1, 2, 3, 4, 5]


#IoU threshold for the detection of strucutre
iou_threshold = 0.5

### Measurements per pixel 

In [ ]:
calculate_metrics_per_pixel(pred_folder = pred_folder, gt_folder = gt_folder, classes = classes)       #list_of_interest = vec[0]['val']    0 is for the fold    #only use list_of_interest for validation (it gives the list of the validation set)

### Measurements per structure/instance

In [ ]:
calculate_metrics_per_structure(pred_folder=pred_folder, gt_folder=gt_folder, classes = classes, iou_threshold = iou_threshold)